
# LLM Core Features with Gemini

Hands-on experiments exploring LLM capabilities using Python and Google's Gemini API.

**Topics covered:**
- Text generation and prompt refinement
- Structured outputs (JSON Schema and Pydantic)
- Image understanding
- Function calling
- Reasoning models
- Embeddings and semantic similarity

**Technologies:** Python, Google Colab, Gemini API, OpenAI-compatible SDK, Google Gen AI SDK.

**Course reference:** Adapted from [The Complete Prompt Engineering for AI Bootcamp (2026) — Core Features Walkthrough](https://github.com/BrightPool/udemy-prompt-engineering-course/blob/main/openai_features_and_functionality/core_features_walkthrough.ipynb).

Selected examples were adapted to Gemini. Some original course features were omitted due to API compatibility and cost considerations.


In [ ]:

%pip install -q -U openai

from openai import OpenAI
import getpass

MODEL = "gemini-3.8-flash"

client = OpenAI(
    api_key=getpass.getpass("Enter Gemini API key: "),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


Enter Gemini API key: ··········



## 1. Text Generation and Prompt Refinement

This example generates a short bedtime story using Gemini.

The second prompt adds explicit requirements for length, audience, setting, tone, and ending to explore how prompt specificity influences the response.


In [ ]:

prompt = "Write a one-sentence bedtime story about a unicorn."

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print("Prompt:", prompt)
print("Response:", response.choices[0].message.content)


Prompt: Write a one-sentence bedtime story about a unicorn.
Response: After an evening of chasing falling stars, the little unicorn curled into a bed of soft clover, dimmed the gentle glow of her horn, and let the moonlit night whisper her to sleep.


In [ ]:

# More specific prompt with explicit requirements
improved_prompt = """
Write a bedtime story about a unicorn.

Requirements:
- Exactly two sentences
- Suitable for children aged 5-7
- Set in a magical forest
- Use simple, comforting language
- End with the unicorn falling asleep peacefully
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": improved_prompt}
    ]
)

print(response.choices[0].message.content)


Deep inside the quiet, glowing forest, a gentle little unicorn curled up on a soft bed of starlit moss. As the silver moon smiled down through the trees, he closed his heavy eyes and drifted off into a sweet, peaceful sleep.



### Observation

The refined prompt produced a story satisfying the specified constraints, demonstrating how clearer instructions can improve control over generated content.

A single successful response does not guarantee consistent results across repeated requests.



## 2. Structured Outputs

Structured outputs allow LLMs to return machine-readable data in a predefined format.

### 2.1 JSON Schema — User Profile Form

This example generates a user profile form as JSON.

JSON Schema defines the expected fields, data types, and required properties, allowing the result to be processed programmatically.


In [ ]:

%pip install -q -U google-genai

from google import genai
import getpass
import json

MODEL = "gemini-3.8-flash"


gemini_client = genai.Client(
    api_key=getpass.getpass("Enter Gemini API key: ")
)


Enter Gemini API key: ··········


In [ ]:

# Define the expected structure of the form
form_schema = {
    "type": "object",
    "properties": {
        "form_title": {
            "type": "string"
        },
        "fields": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "type": {
                        "type": "string",
                        "enum": ["text", "email", "number"]
                    }
                },
                "required": ["name", "type"]
            }
        },
        "submit_button": {
            "type": "string"
        }
    },
    "required": ["form_title", "fields", "submit_button"]
}


In [ ]:

response = gemini_client.interactions.create(
    model=MODEL,
    input="Create a user profile form with name, email, and age fields.",
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": form_schema
    }
)

# Convert the JSON text to a Python dictionary
form_data = json.loads(response.output_text)

# Display the generated form structure
print(json.dumps(form_data, indent=2))


{
  "form_title": "User Profile",
  "fields": [
    {
      "name": "Full Name",
      "type": "text"
    },
    {
      "name": "Email Address",
      "type": "email"
    },
    {
      "name": "Age",
      "type": "number"
    }
  ],
  "submit_button": "Save Profile"
}



### 2.2 Structured Outputs with Pydantic

Pydantic defines and validates data structures using Python classes.

This example generates a step-by-step maths solution containing explanations and a final answer.


In [ ]:

from pydantic import BaseModel

# Structure of one maths step
class Step(BaseModel):
    explanation: str
    output: str

# Structure of the complete maths solution
class MathReasoning(BaseModel):
    steps: list[Step]
    final_answer: str


In [ ]:

response = gemini_client.interactions.create(
    model=MODEL,
    input=(
        "Solve 8x + 7 = -23. "
        "Show the algebraic steps and the final answer."
    ),
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": MathReasoning.model_json_schema()
    }
)

# Validate and convert the JSON into a Pydantic object
math_reasoning = MathReasoning.model_validate_json(
    response.output_text
)

print("Steps:")
for step in math_reasoning.steps:
    print(step.explanation)
    print(step.output)

print("Final answer:", math_reasoning.final_answer)


Steps:
Subtract 7 from both sides of the equation to isolate the term with the variable.
8x = -23 - 7
8x = -30
Divide both sides by 8 to solve for x, then simplify the resulting fraction.
x = -30 / 8
x = -15/4
Final answer: x = -15/4 (or -3.75)



### Key Takeaways

- JSON Schema defines the expected output structure.
- Pydantic validates structured data using Python models.
- Structured outputs are useful for data extraction, automation, and API integration.
- Valid structure does not guarantee factual or mathematical correctness.



## 3. Multimodal Capabilities — Vision

Multimodal models can process images alongside text instructions.

This example sends a JPG image to Gemini and requests a short description of its visual content.

**Image requirement:** The notebook expects a file named `sample.jpg` in the current working directory. Upload the image before running this example.


In [ ]:

import base64

# Read the image
with open("sample.jpg", "rb") as f:
    image_bytes = f.read()

# Send both text and image to Gemini
response = gemini_client.interactions.create(
    model=MODEL,
    input=[
        {
            "type": "text",
            "text": "Describe what you see in this image in 3 sentences."
        },
        {
            "type": "image",
            "data": base64.b64encode(image_bytes).decode("utf-8"),
            "mime_type": "image/jpeg"
        }
    ]
)

print(response.output_text)


A young elephant calf adorably lifts its front foot to scratch or cover its face, with its small trunk curled underneath. It stands sheltered right beside the massive, sturdy legs of an adult elephant, likely its mother. The background features a rocky enclosure wall with twigs and dirt scattered across the ground.



### Observation

Gemini analysed the supplied image and generated a description based on the text instructions.

This demonstrates how prompts can guide visual analysis, including what details to describe and how to present them.



## 4. Function Calling / Tool Calling

Function calling allows an LLM to use external tools through an application.

This example uses a Python function and the Open-Meteo API to retrieve current temperature data.

Gemini selects the function, while the Python application executes it and returns the result.


In [ ]:

import requests

def get_weather(latitude: float, longitude: float) -> float:
    """Get the current temperature in Celsius for given coordinates."""

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m"
    }

    response = requests.get(url, params=params, timeout=15)
    response.raise_for_status()

    data = response.json()

    return data["current"]["temperature_2m"]


In [ ]:

from google.genai import types

response = gemini_client.models.generate_content(
    model=MODEL,
    contents=(
        "What's the current temperature in Paris, France? "
        "Use latitude 48.8566 and longitude 2.3522. "
        "Use the get_weather tool and answer in Celsius."
    ),
    config=types.GenerateContentConfig(
        tools=[get_weather]
    )
)

print(response.text)


The current temperature in Paris, France is 12.6°C.



### Observation

Gemini used the `get_weather()` function to retrieve the temperature in Paris and generated a natural-language answer.

The SDK handled function execution automatically, demonstrating how LLM applications can access external information.



## 5. Reasoning Models

Reasoning-capable models can allocate additional computation to complex tasks.

This example uses a high reasoning level to generate a beginner-friendly explanation of hash tables, collision handling, Python implementation, and time complexity.


In [ ]:

# Request higher reasoning effort
response = gemini_client.interactions.create(
    model=MODEL,
    input=(
        "Explain how to implement a hash table in Python. "
        "Include collision handling, insertion, lookup, "
        "and time complexity. Keep the explanation clear "
        "and beginner-friendly."
    ),
    generation_config={
        "thinking_level": "high"
    }
)

print(response.output_text)


A **hash table** is a data structure that stores data in key-value pairs (just like Python's built-in `dict`). It is designed to make searching, adding, and removing data extremely fast.

Here is a step-by-step, beginner-friendly guide to understanding and building your own hash table in Python from scratch.

---

### How a Hash Table Works

A hash table consists of two main parts:
1. **An internal array (list):** A fixed-size list where the data is actually stored.
2. **A hash function:** A mathematical function that converts a key (e.g., the string `"apple"`) into an integer index corresponding to a slot in our array.

```text
Key: "apple" ──► Hash Function ──► Index: 2 ──► Array[2] = ("apple", 5)
```

---

### Understanding Collisions

Because our array has a fixed size, two completely different keys will eventually produce the exact same index. This is called a **collision**.

*Example:* Both `"apple"` and `"banana"` might hash to index `3`.

#### The Fix: Separate Chaining
The sim


### Observation

Gemini generated a detailed technical explanation, including Python code and complexity analysis.

The prompt controlled the requested content and presentation, while `thinking_level="high"` requested additional reasoning effort.

Higher reasoning effort does not necessarily guarantee better results.



## 6. Embeddings and Semantic Similarity

Embeddings convert text into numerical vectors that represent learned relationships between concepts.

This experiment generates embeddings for 10 words, calculates cosine similarity between every pair, and identifies the most similar words.

**Applications:** Semantic search, document retrieval, clustering, and RAG.


In [ ]:

import numpy as np
from google.genai import types

# Words to compare
words = [
    "king", "queen", "man", "woman",
    "apple", "banana", "orange", "pear",
    "castle", "throne"
]

# Generate embeddings using Gemini
result = gemini_client.models.embed_content(
    model="gemini-embedding-001",
    contents=words,
    config=types.EmbedContentConfig(
        task_type="SEMANTIC_SIMILARITY"
    )
)

# Extract vectors
embeddings = np.array([
    item.values for item in result.embeddings
])

print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", len(embeddings[0]))


Number of embeddings: 10
Embedding dimensions: 3072


In [ ]:

import numpy as np

# Cosine similarity between two vectors
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )

# Compare every word with every other word
similarity_matrix = np.zeros((len(words), len(words)))

for i in range(len(words)):
    for j in range(len(words)):
        similarity_matrix[i, j] = cosine_similarity(
            embeddings[i], embeddings[j]
        )

# Display the complete similarity matrix
print("Similarity Matrix (Cosine Similarity):")
print("          " + " ".join(f"{word:<8}" for word in words))

for i, word in enumerate(words):
    scores = " ".join(
        f"{similarity_matrix[i, j]:<8.4f}"
        for j in range(len(words))
    )
    print(f"{word:<10}{scores}")

# Compare selected relationships
print("\nSpecific relationships:")

pairs = [
    ("king", "queen"),
    ("man", "woman"),
    ("apple", "banana"),
    ("king", "apple"),
    ("castle", "throne")
]

for word1, word2 in pairs:
    i = words.index(word1)
    j = words.index(word2)

    print(f"{word1} vs {word2}: {similarity_matrix[i, j]:.4f}")


Similarity Matrix (Cosine Similarity):
          king     queen    man      woman    apple    banana   orange   pear     castle   throne  
king      1.0000   0.9084   0.8601   0.8080   0.8155   0.8215   0.8210   0.8171   0.8662   0.8962  
queen     0.9084   1.0000   0.8176   0.8753   0.8305   0.8263   0.8259   0.8344   0.8543   0.8960  
man       0.8601   0.8176   1.0000   0.8602   0.8369   0.8416   0.8456   0.8370   0.8464   0.8397  
woman     0.8080   0.8753   0.8602   1.0000   0.8283   0.8349   0.8368   0.8320   0.8345   0.8216  
apple     0.8155   0.8305   0.8369   0.8283   1.0000   0.8837   0.8636   0.9025   0.8179   0.8120  
banana    0.8215   0.8263   0.8416   0.8349   0.8837   1.0000   0.8782   0.8831   0.8059   0.8180  
orange    0.8210   0.8259   0.8456   0.8368   0.8636   0.8782   1.0000   0.8651   0.8267   0.8209  
pear      0.8171   0.8344   0.8370   0.8320   0.9025   0.8831   0.8651   1.0000   0.8235   0.8269  
castle    0.8662   0.8543   0.8464   0.8345   0.8179   0.8059

In [ ]:

for i, word in enumerate(words):
    # Sort other words by similarity, highest first
    ranked = np.argsort(similarity_matrix[i])[::-1]

    # Exclude the word itself
    ranked = [j for j in ranked if j != i][:3]

    print(f"\n{word}:")
    for j in ranked:
        print(f"  {words[j]}: {similarity_matrix[i, j]:.4f}")



king:
  queen: 0.9084
  throne: 0.8962
  castle: 0.8662

queen:
  king: 0.9084
  throne: 0.8960
  woman: 0.8753

man:
  woman: 0.8602
  king: 0.8601
  castle: 0.8464

woman:
  queen: 0.8753
  man: 0.8602
  orange: 0.8368

apple:
  pear: 0.9025
  banana: 0.8837
  orange: 0.8636

banana:
  apple: 0.8837
  pear: 0.8831
  orange: 0.8782

orange:
  banana: 0.8782
  pear: 0.8651
  apple: 0.8636

pear:
  apple: 0.9025
  banana: 0.8831
  orange: 0.8651

castle:
  king: 0.8662
  throne: 0.8600
  queen: 0.8543

throne:
  king: 0.8962
  queen: 0.8960
  castle: 0.8600



### Observation

Cosine similarity was calculated for 10 word embeddings generated using Gemini.

Selected results:
- King–Queen: 0.9084
- Apple–Banana: 0.8837
- King–Apple: 0.8155

Related words generally showed higher similarity, although unrelated words also received relatively high scores.

Similarity values depend on the embedding model and should be interpreted comparatively rather than as percentages of shared meaning.



## Conclusion

These experiments demonstrate how LLMs can generate
text, return structured data, understand images,
use external tools, and represent text as embeddings.

The examples were adapted to Gemini to explore
core AI capabilities using a low-cost API.
